# Dataloaders

In [1]:
import tensorflow as tf
import h5py
import numpy as np
from pathlib import Path

def load_track(file_path, chunk_size=128):
    with h5py.File(file_path, 'r') as f:
        mix = f['mix'][:]
        drums = f['drums'][:]
        bass = f['bass'][:]
        other = f['other'][:]
        vocals = f['vocals'][:]
    
    n_frames = mix.shape[1]
    
    chunks_X = []
    chunks_y = []
    
    for start in range(0, n_frames, chunk_size):
        end = start + chunk_size
        if end > n_frames:
            break
        
        chunk_mix = mix[:, start:end]
        chunk_drums = drums[:, start:end]
        chunk_bass = bass[:, start:end]
        chunk_other = other[:, start:end]
        chunk_vocals = vocals[:, start:end]
        
        chunks_X.append(chunk_mix)
        chunks_y.append(np.stack([chunk_drums, chunk_bass, chunk_other, chunk_vocals], axis=-1))
    
    return chunks_X, chunks_y

def create_dataset(data_path: Path, chunk_size=128, batch_size=8):
    files = list(Path(data_path).glob('*.h5'))
    
    def generator():
        for file in files:
            chunks_X, chunks_y = load_track(file, chunk_size)
            for X, y in zip(chunks_X, chunks_y):
                yield X, y
    
    dataset = tf.data.Dataset.from_generator(generator=generator, output_signature=(
        tf.TensorSpec(shape=(None, chunk_size), dtype=tf.float32),
        tf.TensorSpec(shape=(None, chunk_size, 4), dtype=tf.float32)
    ))
    return dataset.batch(batch_size=batch_size).prefetch(tf.data.AUTOTUNE)

2026-01-13 19:45:11.810722: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-13 19:45:14.645978: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-13 19:45:17.675579: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda-11.8/lib64:
2026-01-13 19:45:17.675709: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: ca

In [2]:
path = Path('../data/processed/train')

dataset = create_dataset(path)

2026-01-13 19:45:23.844748: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-01-13 19:45:24.461959: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-01-13 19:45:24.462167: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-01-13 19:45:24.462655: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags

In [3]:
for X_batch, y_batch in dataset.take(1):
    print("Input shape (mix):", X_batch.shape)
    print("Output shape (stems):", y_batch.shape)
    print("Input range:", X_batch.numpy().min(), "to", X_batch.numpy().max())
    print("Output range:", y_batch.numpy().min(), "to", y_batch.numpy().max())

Input shape (mix): (8, 1025, 128)
Output shape (stems): (8, 1025, 128, 4)
Input range: 3.2965397e-09 to 87.10451
Output range: 1.2259915e-12 to 85.88572


In [4]:
def normalize_example(x, y):
    x_norm = x / tf.reduce_max(tf.abs(x))
    y_norm = y / tf.reduce_max(tf.abs(y), axis=[0,1], keepdims=True)
    return x_norm, y_norm


dataset = dataset.map(normalize_example)
dataset = dataset.shuffle(buffer_size=50)

for X_batch, y_batch in dataset.take(1):
    print("Input shape (mix):", X_batch.shape)
    print("Output shape (stems):", y_batch.shape)
    print("Input range:", X_batch.numpy().min(), "to", X_batch.numpy().max())
    print("Output range:", y_batch.numpy().min(), "to", y_batch.numpy().max())

2026-01-13 19:45:59.590121: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:390] Filling up shuffle buffer (this may take a while): 24 of 50


Input shape (mix): (8, 1025, 128)
Output shape (stems): (8, 1025, 128, 4)
Input range: 4.647526e-11 to 1.0
Output range: 1.2758517e-11 to 1.0


2026-01-13 19:46:14.081298: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:390] Filling up shuffle buffer (this may take a while): 45 of 50
2026-01-13 19:46:14.094163: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:390] Filling up shuffle buffer (this may take a while): 46 of 50
2026-01-13 19:46:14.146529: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:415] Shuffle buffer filled.
